# Build a dartboard wireframe

Explore how the board geometry is estimated and projected onto a photograph. Each image below is calculated from the current input and settings; no pre-generated diagnostic images are loaded.

The automatic overlay is a projection from the fitted homography. It is not ground-truth annotation.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from dartcv.detect import detect_board
from dartcv.diagnostics import detection_overlays
from dartcv.geometry import compute_homography
from dartcv.overlay import draw_overlay, topdown_overlay, warp_topdown

cwd = Path.cwd().resolve()
repo_root = next(path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").is_file())
image_dir = repo_root / "data" / "images" / "raw_board"
image_choices = ["IMG_2086.jpg", "IMG_2092.jpg", "IMG_2100.jpg"]

IMAGE_NAME = "IMG_2092.jpg"
UPRIGHT_DEG = 0.0
REFINE = True
MANUAL_POINTS = None

## Choose an approach

Change `IMAGE_NAME`, `UPRIGHT_DEG`, or `REFINE` above and rerun the image-loading and detection cells. The angle is the prior for which sector points up; set `REFINE = False` to compare the ellipse/bull/orientation estimate before ECC and colour-ring refinement.

For a manual comparison, enter four image-pixel points in `MANUAL_POINTS` in this order: 20|1, 6|10, 3|19, 11|14. Leave it as `None` to skip manual calibration.

In [ ]:
if IMAGE_NAME not in image_choices:
    raise ValueError(f"Choose IMAGE_NAME from {image_choices}")
image_path = image_dir / IMAGE_NAME
image = cv2.imread(str(image_path))
if image is None:
    raise FileNotFoundError(image_path)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
detection = detect_board(
    image,
    upright_deg=UPRIGHT_DEG,
    refine=REFINE,
    diagnostics=True,
)
stage_images = detection_overlays(image, detection)
ring_contours = {name: ring.contour for name, ring in detection.rings.items()}
stage_images["wireframe_overlay.png"] = draw_overlay(
    image,
    detection.H,
    color=(255, 0, 255),
    thickness=3,
    ring_contours=ring_contours,
)
stage_images["topdown.png"] = warp_topdown(image, detection.H)
stage_images["topdown_overlay.png"] = topdown_overlay(
    image, detection.H, ring_contours=ring_contours
)

## Follow the wireframe build-up

Every cell below renders a different stage from the current detection run. Change the settings and rerun the preceding cells to see how the fitted geometry changes.

In [ ]:
def show_stage(filename, caption):
    bgr_image = stage_images.get(filename)
    display(Markdown(f"**{caption}**"))
    if bgr_image is None:
        display(Markdown("This stage is not available with the current settings."))
        return
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

In [ ]:
show_stage("01_colour_mask.png", "1. Red/green colour mask")

In [ ]:
show_stage("02_outer_ellipse.png", "2. Outer boundary samples and ellipse fit")

In [ ]:
show_stage("03_initial_alignment.png", "3. Initial board mapping from ellipse, bull, and orientation")

In [ ]:
show_stage("04_ecc_edges.png", "4. ECC gradient alignment (available when REFINE is enabled)")

In [ ]:
show_stage("05_calibration.png", "5. Projected calibration wires and outer-double intersections")

In [ ]:
show_stage("06_bull_alignment.png", "6. Bull-centre constraint before and after")

In [ ]:
show_stage("07_ring_alignment.png", "7. Colour-band alignment against nominal ring positions")

In [ ]:
show_stage("wireframe_overlay.png", "8. Final board wireframe projected onto the source image")

In [ ]:
show_stage("topdown.png", "9. Source image rectified to a top-down board view")

In [ ]:
show_stage("topdown_overlay.png", "10. Wireframe projected in the rectified view")

## Compare a manual calibration

To try a different approach, fill `MANUAL_POINTS` with the pixel coordinates of the four outer-double wire intersections in order: 20|1, 6|10, 3|19, 11|14. Rerun this cell to project the same board model from those supplied points.

In [ ]:
if MANUAL_POINTS is None:
    display(Markdown("Set `MANUAL_POINTS` above to four image-pixel coordinates, then rerun this cell to compare a manual wireframe."))
else:
    manual_H = compute_homography(MANUAL_POINTS)
    manual_overlay = draw_overlay(
        image, manual_H, color=(255, 0, 255), thickness=3
    )
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(manual_overlay, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()